# 04. Validar lista ouro (perspectiva Censo)

A lista (`cohort_dedup`) diz: registro Censo **A** deve receber o **CPF X**.
Aqui medimos o resultado operacional nos **clusters a 0,95** (não nos pares
entre 0,5 e 0,95).

**Acerto:** o cluster de A contém X. Outros Censos no mesmo cluster são ok.

Funil:

1. Censos no subset
2. Censos cujo cluster tem ≥1 CPF (`n_censo_com_match`)
3. Censos ouro avaliáveis (A e X no subset)
4. Ouro recuperado (X no cluster de A) → **recall_ouro = (4)/(3)**

O recall por `cluster_id` infla se houver mega-clusters N×M — por isso a
quebra por tipo (`1_para_1`, `1_cpf_n_censo`, `outros`).

Labels Splink (P/R, FP/FN): [`03_validar_coorte.ipynb`](03_validar_coorte.ipynb).

**Pré-requisito:** NB02 (`splink_clusters.parquet`) e `registro_limpo`.

In [ ]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from IPython.display import display

from config import (
    COHORT_DEDUP_ARQUIVO,
    METRICAS_OURO,
    SPLINK_CLUSTERS,
    SPLINK_INPUT_VIEW,
    TABELA_LIMPA,
    cpf_norm_sql,
    drop_splink_temp_tables,
    get_connection,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

THRESHOLD_AVALIACAO = 0.95
TOP_N_CLUSTERS = 10

print_paths()
require_input(COHORT_DEDUP_ARQUIVO, label='COHORT')
require_input(SPLINK_CLUSTERS, label='SPLINK_CLUSTERS (rode o NB02 antes)')

con = get_connection()
drop_splink_temp_tables(con)
require_tables(con, [TABELA_LIMPA], notebook_origem='00b')
materialize_splink_input(con)

con.execute(f"""
CREATE OR REPLACE TABLE cohort_dedup_raw AS
SELECT * FROM read_parquet('{COHORT_DEDUP_ARQUIVO}')
""")
print('Registros na coorte:', con.execute('SELECT COUNT(*) FROM cohort_dedup_raw').fetchone()[0])
print('Threshold (clusters):', THRESHOLD_AVALIACAO)

## 1. Ouro 1:1 no subset

Só pares estritamente 1:1 (um Censo ↔ um CPF) e com **os dois lados** no
recorte. Cada linha é um Censo A que deveria receber o CPF X.

In [ ]:
CPF_GT = cpf_norm_sql('CPF_NORM')
con.execute(f'''
CREATE OR REPLACE TABLE cohort_pares AS
SELECT DISTINCT
    CAST(PERSON_ID_CENSO AS VARCHAR) AS person_id_censo,
    {CPF_GT} AS cpf_norm
FROM cohort_dedup_raw
WHERE PERSON_ID_CENSO IS NOT NULL AND CPF_NORM IS NOT NULL
''')

con.execute('''
CREATE OR REPLACE TABLE ground_truth_pairs AS
SELECT
    'censo_' || person_id_censo AS unique_id_censo,
    'cpf_' || cpf_norm AS unique_id_cpf,
    person_id_censo,
    cpf_norm
FROM (
    SELECT *,
        COUNT(*) OVER (PARTITION BY person_id_censo) AS n_cpf_por_censo,
        COUNT(*) OVER (PARTITION BY cpf_norm) AS n_censo_por_cpf
    FROM cohort_pares
)
WHERE n_cpf_por_censo = 1 AND n_censo_por_cpf = 1
''')

con.execute(f'''
CREATE OR REPLACE TABLE gt_no_subset AS
SELECT gt.*
FROM ground_truth_pairs gt
JOIN {SPLINK_INPUT_VIEW} c ON c.unique_id = gt.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} p ON p.unique_id = gt.unique_id_cpf
''')

display(con.execute('''
SELECT
    (SELECT COUNT(*) FROM ground_truth_pairs) AS n_pares_1a1_nacional,
    (SELECT COUNT(*) FROM gt_no_subset) AS n_ouro_subset
''').df())

## 2. Composição dos clusters (0,95)

`1_para_1` e `1_cpf_n_censo` são os casos naturais. `outros` (N CPFs × M Censos)
e mega-clusters inflacionam o recall por `cluster_id`.

In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE splink_clusters AS
SELECT * FROM read_parquet('{SPLINK_CLUSTERS}')
''')

con.execute(f'''
CREATE OR REPLACE TABLE cluster_composicao AS
SELECT
    cluster_id,
    COUNT(*) AS n,
    COUNT(*) FILTER (WHERE s.origem = 'censo') AS n_censo,
    COUNT(*) FILTER (WHERE s.origem = 'cpf') AS n_cpf,
    CASE
        WHEN COUNT(*) = 1 THEN 'singleton'
        WHEN COUNT(*) FILTER (WHERE s.origem = 'censo') = 1
         AND COUNT(*) FILTER (WHERE s.origem = 'cpf') = 1 THEN '1_para_1'
        WHEN COUNT(*) FILTER (WHERE s.origem = 'cpf') = 1
         AND COUNT(*) FILTER (WHERE s.origem = 'censo') >= 2 THEN '1_cpf_n_censo'
        ELSE 'outros'
    END AS tipo
FROM splink_clusters sc
JOIN {SPLINK_INPUT_VIEW} s ON s.unique_id = sc.unique_id
GROUP BY cluster_id
''')

display(con.execute('''
SELECT tipo, COUNT(*) AS n_clusters, SUM(n) AS n_registros,
       SUM(n_censo) AS n_censo, SUM(n_cpf) AS n_cpf
FROM cluster_composicao
GROUP BY tipo
ORDER BY n_clusters DESC
''').df())

## 3. Funil na perspectiva do Censo

`n_censo_com_match` = Censo cujo cluster (0,95) contém pelo menos um CPF.
`n_ouro_recuperado` = Censo ouro A cujo cluster contém o CPF ouro X.
**recall_ouro** = recuperados / ouro no subset.

In [ ]:
funil = con.execute(f'''
WITH censo_all AS (
    SELECT unique_id
    FROM {SPLINK_INPUT_VIEW}
    WHERE origem = 'censo'
),
censo_cluster AS (
    SELECT c.unique_id, sc.cluster_id, comp.tipo, comp.n_cpf
    FROM censo_all c
    LEFT JOIN splink_clusters sc ON sc.unique_id = c.unique_id
    LEFT JOIN cluster_composicao comp ON comp.cluster_id = sc.cluster_id
),
linkados AS (
    SELECT unique_id
    FROM censo_cluster
    WHERE n_cpf >= 1
),
ouro_hit AS (
    SELECT
        gt.unique_id_censo,
        CASE WHEN cl.cluster_id IS NOT NULL AND cl.cluster_id = cr.cluster_id
             THEN 1 ELSE 0 END AS hit,
        COALESCE(comp.tipo, 'sem_cluster') AS tipo_cluster
    FROM gt_no_subset gt
    LEFT JOIN splink_clusters cl ON cl.unique_id = gt.unique_id_censo
    LEFT JOIN splink_clusters cr ON cr.unique_id = gt.unique_id_cpf
    LEFT JOIN cluster_composicao comp ON comp.cluster_id = cl.cluster_id
)
SELECT
    (SELECT COUNT(*) FROM censo_all) AS n_censo_subset,
    (SELECT COUNT(*) FROM linkados) AS n_censo_com_match,
    ROUND(
        100.0 * (SELECT COUNT(*) FROM linkados)
        / NULLIF((SELECT COUNT(*) FROM censo_all), 0),
        2
    ) AS pct_censo_linkado,
    (SELECT COUNT(*) FROM gt_no_subset) AS n_ouro,
    (SELECT SUM(hit) FROM ouro_hit) AS n_ouro_recuperado,
    ROUND(
        1.0 * (SELECT SUM(hit) FROM ouro_hit)
        / NULLIF((SELECT COUNT(*) FROM gt_no_subset), 0),
        4
    ) AS recall_ouro,
    (
        SELECT COUNT(*)
        FROM linkados l
        JOIN gt_no_subset g ON g.unique_id_censo = l.unique_id
    ) AS n_linkados_no_ouro
''').df()
funil['threshold_avaliacao'] = THRESHOLD_AVALIACAO
display(funil)

print(
    f"Dos {int(funil.n_censo_subset.iloc[0]):,} Censos no subset, "
    f"{int(funil.n_censo_com_match.iloc[0]):,} ({funil.pct_censo_linkado.iloc[0]}%) "
    f"caíram num cluster com ≥1 CPF.\n"
    f"Desses linkados, {int(funil.n_linkados_no_ouro.iloc[0]):,} estavam na lista ouro.\n"
    f"Ouro no subset: {int(funil.n_ouro.iloc[0]):,}; recuperados: "
    f"{int(funil.n_ouro_recuperado.iloc[0]):,} → recall_ouro = {funil.recall_ouro.iloc[0]}"
)

## 4. Recall ouro por tipo de cluster

Se `outros` concentrar a maior parte dos hits, o número agregado está inflado
por over-clustering.

In [ ]:
display(con.execute('''
SELECT
    COALESCE(comp.tipo, 'sem_cluster') AS tipo_cluster,
    COUNT(*) AS n_censo_ouro,
    SUM(CASE WHEN cl.cluster_id = cr.cluster_id THEN 1 ELSE 0 END) AS hits,
    ROUND(
        1.0 * SUM(CASE WHEN cl.cluster_id = cr.cluster_id THEN 1 ELSE 0 END)
        / NULLIF(COUNT(*), 0),
        4
    ) AS recall
FROM gt_no_subset gt
LEFT JOIN splink_clusters cl ON cl.unique_id = gt.unique_id_censo
LEFT JOIN splink_clusters cr ON cr.unique_id = gt.unique_id_cpf
LEFT JOIN cluster_composicao comp ON comp.cluster_id = cl.cluster_id
GROUP BY 1
ORDER BY n_censo_ouro DESC
''').df())

## 5. Top 10 clusters (o que foi unificado)

Os maiores — para ver homônimos, CEP compartilhado ou over-clustering.

In [ ]:
top = con.execute(f'''
WITH top_ids AS (
    SELECT cluster_id, n, tipo, n_censo, n_cpf
    FROM cluster_composicao
    ORDER BY n DESC
    LIMIT {TOP_N_CLUSTERS}
)
SELECT
    t.cluster_id, t.n, t.tipo, t.n_censo, t.n_cpf,
    s.unique_id, s.origem, s.nome_completo, s.data_nascimento, s.sexo, s.cep
FROM top_ids t
JOIN splink_clusters sc ON sc.cluster_id = t.cluster_id
JOIN {SPLINK_INPUT_VIEW} s ON s.unique_id = sc.unique_id
ORDER BY t.n DESC, t.cluster_id, s.origem, s.unique_id
''').df()
display(
    top.groupby(['cluster_id', 'n', 'tipo', 'n_censo', 'n_cpf'])
    .size().rename('membros').reset_index()
    .sort_values('n', ascending=False)
)
for cid, g in top.groupby('cluster_id', sort=False):
    display(g.drop(columns=['n', 'tipo', 'n_censo', 'n_cpf']).head(30))

In [ ]:
funil.to_csv(METRICAS_OURO, index=False)
print('Métricas ouro salvas:', METRICAS_OURO)
con.close()